## Imports

In [1]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
import spacy
! python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 90.9 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


In [3]:
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])

## Loading the metadata file

In [4]:
# Load the CSV data - metadata
df = pd.read_csv('../data/archelect_search.csv')

# Extract year and filter
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
df = df[df['contexte-election'] == "législatives"]

## Extracting texts and joining w/ metadata file

In [5]:
# Extract texts from text files 
records = []

for root, dirs, files in os.walk('../text_files'):
    for file in files:
        if file.endswith('.txt') :
            file_id = file[:-4]

            if file_id in df['id'].values:
                file_path = os.path.join(root, file)
            
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    
                    records.append({
                        "id": file_id,
                        "text": content,
                        "file_name": file
                    })
                    
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

df_texts = pd.DataFrame(records)

In [6]:
# Merge w/ metadata
df_final = df.merge(df_texts, on="id", how="inner")

print(f"Total number of documents : {len(df_final)}")

# Stats
lengths = df_final["text"].apply(lambda x: len(x.split()))
print(f"Average length: {lengths.mean():.0f}")
print(f"Min: {lengths.min()}")
print(f"Max: {lengths.max()}")

Total number of documents : 21167
Average length: 712
Min: 19
Max: 4002


## Lemmatization

In [7]:
df_final['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(df_final['text'])]

In [8]:
df_final.head()

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,year,text,file_name,lemmatized_text
0,EL065_L_1973_03_001_01_1_PF_01,1973-03-04,Assemblée Nationale;France;Ve République;Élect...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,EL065_L_1973_03_001_01_1_PF_01.txt,science po / fonds cevipof \n republique franç...
1,EL065_L_1973_03_001_01_1_PF_02,1973-03-04,Élections législatives;Ve République;Assemblée...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,non mentionné,non mentionné,Parti socialiste;Mouvement des radicaux de gauche,Union de la gauche socialiste et démocrate,non,1973,REPUBLIQUE FRANCAISE - LIBERTE - EGALITE - FRA...,EL065_L_1973_03_001_01_1_PF_02.txt,republique FRANCAISE - LIBERTE - EGALITE - fra...
2,EL065_L_1973_03_001_01_1_PF_03,1973-03-04,Assemblée Nationale;Ve République;Élections lé...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,non mentionné,non mentionné,non mentionné,Faîtes confiance aux jeunes d'aujourd'hui,non,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,EL065_L_1973_03_001_01_1_PF_03.txt,science po / fonds cevipof \n republique franç...
3,EL065_L_1973_03_001_01_1_PF_04,1973-03-04,France;Ve République;Élections législatives;As...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,non mentionné,non mentionné,Parti communiste français,Union populaire et victoire du programme commun,oui,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,EL065_L_1973_03_001_01_1_PF_04.txt,science po / fonds cevipof \n republique franç...
4,EL065_L_1973_03_001_01_1_PF_05,1973-03-04,Élections législatives;France;Assemblée Nation...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,sports et loisirs,résistant,non mentionné,Majorité Ve République,oui,1973,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,EL065_L_1973_03_001_01_1_PF_05.txt,science po / fonds cevipof \n republique franç...


## Save the complete dataframe 

In [9]:
df_final.to_pickle("../data/df_final.pkl")

## Saving on S3 

In [10]:
import s3fs
import pickle

BUCKET = "victorgalmiche"
FILE   = "projet-nlp/df_final.pkl"

fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': 'https://minio.lab.sspcloud.fr'})

with fs.open(f"{BUCKET}/{FILE}", "wb") as f:
    pickle.dump(df_final, f)

In [ ]:
# Testing that it loads well
with fs.open(f"{BUCKET}/{FILE}", "rb") as f:
    dataframe = pickle.load(f)